# RFM Customer Segmentation

The **objective** is to segment customers based on
their purchasing behavior using RFM analysis.

RFM represents:

- Recency — how recently a customer purchased
- Frequency — how often a customer purchased
- Monetary — how much a customer spent

The resulting RFM scores and segments will be used as a foundation
for the later customer retention and churn-risk analysis.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)

sns.set_theme(style="whitegrid")

In [4]:
# Import the data
PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

sales_file = PROCESSED_DIR / "retail_sales_cleaned.csv"

sales_df = pd.read_csv(sales_file, parse_dates=["InvoiceDate"])

print(f"Sales Dataset loaded: {sales_df.shape}")

C:\Users\tanya\AppData\Local\Temp\ipykernel_16840\3663297509.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_df = pd.read_csv(sales_file, parse_dates=["InvoiceDate"])


Sales Dataset loaded: (524878, 10)


In [7]:
# Check the transaction observation period
min_date = sales_df["InvoiceDate"].min()
max_date = sales_df["InvoiceDate"].max()

print(f"First transaction: {min_date}")
print(f"Last transaction:  {max_date}")

First transaction: 2010-12-01 08:26:00
Last transaction:  2011-12-09 12:50:00


### Establishing the RFM Observation Period

RFM recency must be calculated relative to the end of the available
customer observation period.

The dataset's final recorded transaction occurred on December 9, 2011.

For RFM analysis, **December 10, 2011** is used as the reference date.

In [8]:
# Define the RFM reference date
reference_date = pd.Timestamp("2011-12-10")
print(f"RFM Reference Date: {reference_date}")

RFM Reference Date: 2011-12-10 00:00:00


## Recency = Reference Date - Last Purchase Date

In [9]:
# Calculate each customer's most recent purchase date
last_purchase = (
    sales_df.groupby("CustomerID")["InvoiceDate"]
    .max()
    .reset_index()
    .rename(columns={"InvoiceDate": "Last_Purchase_Date"})
)
last_purchase.head()

,CustomerID,Last_Purchase_Date
0,12346.0,2011-01-18 10:01:00
1,12347.0,2011-12-07 15:52:00
2,12348.0,2011-09-25 13:13:00
3,12349.0,2011-11-21 09:51:00
4,12350.0,2011-02-02 16:01:00


In [10]:
# Calculate Recency in days
last_purchase["Recency"] = (
    reference_date - last_purchase["Last_Purchase_Date"]
).dt.days

last_purchase.head()

,CustomerID,Last_Purchase_Date,Recency
0,12346.0,2011-01-18 10:01:00,325
1,12347.0,2011-12-07 15:52:00,2
2,12348.0,2011-09-25 13:13:00,75
3,12349.0,2011-11-21 09:51:00,18
4,12350.0,2011-02-02 16:01:00,310


In [11]:
last_purchase["Recency"].describe()

count    4338.000000
mean       92.059474
std       100.012264
min         0.000000
25%        17.000000
50%        50.000000
75%       141.750000
max       373.000000
Name: Recency, dtype: float64

- 25% of customers purchased within the last 17 days of the observation period.
- 50% of customers purchased within 50 days.
- 75% purchased within about 142 days.
- Some customers had not purchased for as long as 373 days.

## Frequency = No. of distinct orders(invoices) made by each customer

In [12]:
# Frequency must count actual orders, not transaction lines
# How many times did this customer actually place an order?
frequency = (
    sales_df.groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .reset_index()
    .rename(columns={"InvoiceNo": "Frequency"})
)
frequency.head()

,CustomerID,Frequency
0,12346.0,1
1,12347.0,7
2,12348.0,4
3,12349.0,1
4,12350.0,1


In [13]:
frequency["Frequency"].describe()

count    4338.000000
mean        4.272245
std         7.697975
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       209.000000
Name: Frequency, dtype: float64

## Monetary = Total amount spent by each customer

In [14]:
monetary = (
    sales_df.groupby("CustomerID")["Sales"]
    .sum()
    .reset_index()
    .rename(columns={"Sales": "Monetary"})
)
monetary.head()

,CustomerID,Monetary
0,12346.0,77183.60
1,12347.0,4310.00
2,12348.0,1797.24
3,12349.0,1757.55
4,12350.0,334.40


In [16]:
monetary["Monetary"].describe()

count      4338.000000
mean       2048.688081
std        8985.230220
min           3.750000
25%         306.482500
50%         668.570000
75%        1660.597500
max      280206.020000
Name: Monetary, dtype: float64

In [17]:
# Combine R, F and M
rfm = (
    last_purchase[["CustomerID", "Recency"]]
    .merge(frequency, on="CustomerID", how="inner")
    .merge(monetary, on="CustomerID", how="inner")
)
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,325,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,18,1,1757.55
4,12350.0,310,1,334.40


## RFM Scoring
```
                   1        2        3        4        5
Recency          Worst ----------------------------> Best
                 high days                         low days

Frequency        Lowest --------------------------> Highest

Monetary         Lowest --------------------------> Highest

```

### Quintile = divide your customers into 5 groups based on their value.
- Gives five behavior-based groups of approximately equal size, allowing customers to be ranked relative to the rest of the customer base.

In [20]:
# Recency quintile scoring
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm[["CustomerID", "Recency", "R_Score"]].head()

,CustomerID,Recency,R_Score
0,12346.0,325,1
1,12347.0,2,5
2,12348.0,75,2
3,12349.0,18,4
4,12350.0,310,1


- ```qcut``` is a pandas function that divides your data into a specified number of groups based on percentiles.

Why [5,4,3,2,1]?

- For Recency, we reversed the labels.
- lowest recency -> 5
- highest recency -> 1

In [21]:
rfm["R_Score"].value_counts().sort_index()

R_Score
1    861
2    866
3    863
4    880
5    868
Name: count, dtype: int64

In [23]:
# Frequency quintile scorig with rank
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"), #if 2 customers have the same freq, rank them according to
    q=5,                                   # to the order in which they appear in the df
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm[["CustomerID", "Frequency", "F_Score"]].head()

,CustomerID,Frequency,F_Score
0,12346.0,1,1
1,12347.0,7,5
2,12348.0,4,4
3,12349.0,1,1
4,12350.0,1,1


In [24]:
rfm["F_Score"].value_counts().sort_index()

F_Score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

In [26]:
# Monetary scoring
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm[["CustomerID", "Monetary", "M_Score"]].head()

,CustomerID,Monetary,M_Score
0,12346.0,77183.60,5
1,12347.0,4310.00,5
2,12348.0,1797.24,4
3,12349.0,1757.55,4
4,12350.0,334.40,2


In [27]:
rfm['M_Score'].value_counts().sort_index()

M_Score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

## Create the combined RFM score

RFM Score = R_Score + F_Score + M_Score
- Minimum = 1 + 1 + 1 = 3  (weak RFM profile)
- Maximum = 5 + 5 + 5 = 15  (excellent customer profile)

In [29]:
rfm["RFM_Score"] = (
    rfm["R_Score"]
    + rfm["F_Score"]
    + rfm["M_Score"]
)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346.0,325,1,77183.60,1,1,5,7
1,12347.0,2,7,4310.00,5,5,5,15
2,12348.0,75,4,1797.24,2,4,4,10
3,12349.0,18,1,1757.55,4,1,4,9
4,12350.0,310,1,334.40,1,1,2,4


In [32]:
rfm["RFM_Score"].value_counts().sort_index()

RFM_Score
3     182
4     363
5     338
6     427
7     382
8     374
9     332
10    344
11    349
12    316
13    286
14    298
15    347
Name: count, dtype: int64

## RFM Customer Segmentation

Customers are assigned to behavioral segments using their Recency,
Frequency and Monetary scores.

The segments are designed to support customer-value interpretation
and the later retention/churn analysis.

- **Champions:** high Recency, Frequency and Monetary scores
- **Loyal / Valuable:** strong repeat purchasing and customer value
- **Recent Customers:** recent purchases but lower frequency
- **At Risk:** low Recency score combined with stronger historical
  frequency or monetary value
- **Hibernating / Low Engagement:** low scores across all three
  dimensions
- **Other:** customers whose RFM profile does not fit the primary
  behavioral categories

"At Risk" at this stage is an RFM behavioral segment. It is not the
final churn prediction target.

In [36]:
def assign_rfm_segment(row):
    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"

    elif r >= 3 and f >= 4 and m >= 3:
        return "Loyal / Valuable"

    elif r >= 4 and f <= 3:
        return "Recent Customers"

    elif r <= 2 and (f >= 3 or m >= 3):
        return "At Risk"

    elif r <= 2 and f <= 2 and m <= 2:
        return "Hibernating / Low Engagement"

    else:
        return "Other"


rfm["RFM_Segment"] = rfm.apply(assign_rfm_segment, axis=1)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Segment
0,12346.0,325,1,77183.60,1,1,5,7,At Risk
1,12347.0,2,7,4310.00,5,5,5,15,Champions
2,12348.0,75,4,1797.24,2,4,4,10,At Risk
3,12349.0,18,1,1757.55,4,1,4,9,Recent Customers
4,12350.0,310,1,334.40,1,1,2,4,Hibernating / Low Engagement


In [37]:
rfm["RFM_Segment"].value_counts()

RFM_Segment
Champions                       949
At Risk                         903
Hibernating / Low Engagement    824
Recent Customers                619
Other                           587
Loyal / Valuable                456
Name: count, dtype: int64

In [38]:
# Compare segment behavior
segment_profile = (
    rfm.groupby("RFM_Segment")
    .agg(
        Customers=("CustomerID", "nunique"),
        Avg_Recency=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Monetary=("Monetary", "mean"),
        Avg_RFM_Score=("RFM_Score", "mean")
    )
    .sort_values("Avg_RFM_Score", ascending=False)
)

segment_profile.round(2)

,Customers,Avg_Recency,Avg_Frequency,Avg_Monetary,Avg_RFM_Score
RFM_Segment,,,,,
Champions,949,12.12,11.16,6064.02,13.93
Loyal / Valuable,456,38.17,5.28,1976.16,11.54
Recent Customers,619,16.67,1.79,877.46,9.04
At Risk,903,159.23,2.83,1274.19,7.85
Other,587,49.79,1.73,594.84,7.57
Hibernating / Low Engagement,824,227.09,1.04,228.66,4.19


In [39]:
# Save RFM customer-level dataset
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

Output directory: C:\Users\tanya\Retail_Sales_Customer_Analytics\data\processed


In [40]:
rfm_file = OUTPUT_DIR / "customer_rfm.csv"

rfm.to_csv(rfm_file, index=False)

print(f"RFM dataset saved to: {rfm_file}")
print(f"Shape: {rfm.shape}")

RFM dataset saved to: C:\Users\tanya\Retail_Sales_Customer_Analytics\data\processed\customer_rfm.csv
Shape: (4338, 9)
